# T6 — Agency & Influence Correlation Analysis

Examines whether participants' sense of agency relates to how much each audio manipulation influenced their gait. The analysis runs identically for both conditions.

**Research question:** Do participants who felt more in control of their walking show stronger or weaker gait adaptation to the audio cues?

## Variables

| Variable | Description |
|---|---|
| `agency_q1/q2/q3` | Individual agency Likert items (1–10) |
| `agency_aggregate` | Mean of the three items |
| `ueqs_pragmatic` | UEQ-S pragmatic quality subscale |
| `ari_immersion` | ARI immersion score |
| `influence_score` | Continuous gait-change score (0–1) |
| `category` | Response profile: *Consistent influence*, *Subconscious correction*, *No clear effect*, *Impact only from drift phases* |
| `direction` | Manipulation direction (condition-specific) |

## Analysis steps (per condition)
1. Descriptive statistics and distributions
2. Spearman correlation matrix across all agency/experience metrics
3. Scatter plots — agency aggregate vs influence score, split by direction
4. Group comparison — agency scores by influence category (boxplots + Kruskal-Wallis)
5. Individual agency items vs influence score
6. Cross-condition comparison (weight vs tempo, within-subject)

In [ ]:
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 14})

DATA_DIR   = Path(".")
EXPORT_DIR = DATA_DIR / "exports"
FIG_DIR    = EXPORT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

# ── Load influence tables ──────────────────────────────────────────────────────
df_weight = pd.read_csv(EXPORT_DIR / "weight_influence_table.csv")
df_weight["pid"] = df_weight["pid"].astype(str).str.zfill(2)

df_tempo = pd.read_csv(EXPORT_DIR / "tempo_influence_table.csv")
df_tempo["pid"] = df_tempo["pid"].astype(str).str.zfill(2)

# ── Load agency / experience metrics from summary spreadsheet ─────────────────
_summary = pd.read_excel(DATA_DIR / "Test 6_summary.xlsx", header=2)
_summary["pid"] = _summary["ID"].astype(str).str.zfill(2)
_summary = _summary.set_index("pid")

for df_cond in [df_weight, df_tempo]:
    for local_col, src_col in [
        ("agency_aggregate", "agency aggregate"),
        ("ueqs_pragmatic",   "ueqs pragmatic"),
        ("ari_immersion",    "ari immersion"),
        ("agency_q1",        "agency q1"),
        ("agency_q2",        "agency q2"),
        ("agency_q3",        "agency q3"),
    ]:
        if src_col in _summary.columns:
            df_cond[local_col] = df_cond["pid"].map(_summary[src_col])

# ── Shared constants ──────────────────────────────────────────────────────────
CATEGORY_ORDER = [
    "No clear effect",
    "Impact only from drift phases",
    "Subconscious correction",
    "Consistent influence",
]
CATEGORY_COLORS = {
    "No clear effect":               "#7f8c8d",
    "Impact only from drift phases": "#e67e22",
    "Subconscious correction":       "#3498db",
    "Consistent influence":          "#27ae60",
}
VAR_LABELS = {
    "agency_q1":        "Agency Q1",
    "agency_q2":        "Agency Q2",
    "agency_q3":        "Agency Q3",
    "agency_aggregate": "Agency (agg)",
    "ueqs_pragmatic":   "UEQ-S Prag.",
    "ari_immersion":    "ARI Imm.",
    "influence_score":  "Influence",
}
SHORT_CAT = {
    "No clear effect":               "No effect",
    "Impact only from drift phases": "Drift phases",
    "Subconscious correction":       "Subconscious",
    "Consistent influence":          "Consistent",
}

def agency_cols_for(df_cond):
    candidates = ["agency_q1", "agency_q2", "agency_q3",
                  "agency_aggregate", "ueqs_pragmatic", "ari_immersion"]
    return [c for c in candidates if c in df_cond.columns and df_cond[c].notna().any()]

print(f"Weight: {len(df_weight)} participants  |  Tempo: {len(df_tempo)} participants")
print(f"Weight agency cols: {agency_cols_for(df_weight)}")
print(f"Tempo  agency cols: {agency_cols_for(df_tempo)}")

In [ ]:
def analyse_condition(df_cond, condition_name, direction_values, fig_prefix):
    """
    Run the full agency-vs-influence analysis for one manipulation condition.

    Parameters
    ----------
    df_cond          : DataFrame with columns influence_score, agency_*, category, direction, pid
    condition_name   : string label for plot titles (e.g. 'Weight' or 'Tempo')
    direction_values : the two direction strings used in this condition
    fig_prefix       : filename prefix for saved figures (e.g. 'weight' or 'tempo')
    """
    acols = agency_cols_for(df_cond)

    # ── 1. Summary statistics ─────────────────────────────────────────────────
    extra = [c for c in ["pct_achieved"] if c in df_cond.columns]
    desc = df_cond[acols + ["influence_score"] + extra].describe().T
    desc.index.name = "variable"
    print(f"\n=== {condition_name} condition: descriptive statistics ===")
    display(desc.round(3))

    # ── 2. Spearman correlation matrix ────────────────────────────────────────
    corr_vars = acols + ["influence_score"]
    n = len(corr_vars)
    rho_mat  = pd.DataFrame(index=corr_vars, columns=corr_vars, dtype=float)
    pval_mat = pd.DataFrame(index=corr_vars, columns=corr_vars, dtype=float)
    for v1 in corr_vars:
        for v2 in corr_vars:
            valid = df_cond[[v1, v2]].dropna()
            if len(valid) < 5 or v1 == v2:
                rho_mat.loc[v1, v2]  = 1.0 if v1 == v2 else np.nan
                pval_mat.loc[v1, v2] = np.nan
                continue
            rho, p = stats.spearmanr(valid[v1], valid[v2])
            rho_mat.loc[v1, v2], pval_mat.loc[v1, v2] = rho, p

    fig, ax = plt.subplots(figsize=(max(5, n * 1.1), max(4, n * 0.9)))
    rho_arr = rho_mat.values.astype(float)
    im = ax.imshow(rho_arr, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
    plt.colorbar(im, ax=ax, label="Spearman ρ")
    tick_labels = [VAR_LABELS.get(v, v) for v in corr_vars]
    ax.set_xticks(range(n)); ax.set_xticklabels(tick_labels, rotation=35, ha="right")
    ax.set_yticks(range(n)); ax.set_yticklabels(tick_labels)
    for i in range(n):
        for j in range(n):
            r, p = rho_arr[i, j], pval_mat.values[i, j]
            if np.isnan(r): continue
            sig = "**" if (not np.isnan(p) and p < 0.01) else ("*" if (not np.isnan(p) and p < 0.05) else "")
            ax.text(j, i, f"{r:.2f}{sig}", ha="center", va="center",
                    color="white" if abs(r) > 0.5 else "black")
    ax.set_title(f"{condition_name} — Spearman correlation matrix\n(* p<.05, ** p<.01)")
    plt.tight_layout(); plt.show()
    fig.savefig(FIG_DIR / f"{fig_prefix}_correlation_matrix.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"\nCorrelations with influence_score ({condition_name}):")
    for v in acols:
        valid = df_cond[[v, "influence_score"]].dropna()
        rho, p = stats.spearmanr(valid[v], valid["influence_score"])
        sig = "**" if p < 0.01 else ("*" if p < 0.05 else "†" if p < 0.10 else "ns")
        print(f"  {VAR_LABELS.get(v, v):<22s}  ρ = {rho:+.3f}  p = {p:.3f}  {sig}  (n={len(valid)})")

    # ── 3. Scatter plots: agency aggregate vs influence score ─────────────────
    if "agency_aggregate" not in acols:
        print("agency_aggregate not available — skipping scatter plots.")
    else:
        subsets = [
            ("All participants",         df_cond),
            (f"{direction_values[0].capitalize()}",
             df_cond[df_cond["direction"] == direction_values[0]]),
            (f"{direction_values[1].capitalize()}",
             df_cond[df_cond["direction"] == direction_values[1]]),
        ]
        fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
        fig.suptitle(f"{condition_name}: agency aggregate vs influence score")
        for ax, (title, sub) in zip(axes, subsets):
            sub = sub.dropna(subset=["agency_aggregate", "influence_score"])
            if len(sub) < 3:
                ax.set_title(title + " (too few)"); continue
            for cat, grp in sub.groupby("category"):
                ax.scatter(grp["agency_aggregate"], grp["influence_score"],
                           color=CATEGORY_COLORS.get(cat, "#95a5a6"),
                           label=cat, s=60, zorder=3, edgecolors="white", linewidths=0.5)
            for _, row in sub.iterrows():
                ax.annotate(row["pid"], (row["agency_aggregate"], row["influence_score"]),
                            fontsize=10, ha="left", va="bottom",
                            xytext=(3, 2), textcoords="offset points", color="#555")
            x, y = sub["agency_aggregate"].values, sub["influence_score"].values
            m, b, r, p_lin, _ = stats.linregress(x, y)
            xfit = np.linspace(x.min(), x.max(), 100)
            ax.plot(xfit, m * xfit + b, color="#2c3e50", lw=1.2, linestyle="--",
                    label=f"r={r:.2f}, p={p_lin:.3f}")
            rho, p_sp = stats.spearmanr(x, y)
            ax.set_title(f"{title}\nSpearman ρ={rho:.2f}, p={p_sp:.3f}")
            ax.set_xlabel("Agency aggregate")
            ax.legend(fontsize=10, frameon=True, loc="upper right")
        axes[0].set_ylabel("Influence score")
        plt.tight_layout(); plt.show()
        fig.savefig(FIG_DIR / f"{fig_prefix}_scatter.png", dpi=150, bbox_inches="tight")
        plt.close(fig)

    # ── 4. Group comparison: agency by influence category ─────────────────────
    metrics = {k: v for k, v in {
        "agency_aggregate": "Agency (aggregate)",
        "ueqs_pragmatic":   "UEQ-S Pragmatic",
        "ari_immersion":    "ARI Immersion",
    }.items() if k in df_cond.columns and df_cond[k].notna().any()}

    if metrics:
        fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 5))
        if len(metrics) == 1: axes = [axes]
        fig.suptitle(f"{condition_name}: agency scores by influence category")
        for ax, (col, label) in zip(axes, metrics.items()):
            groups = [df_cond[df_cond["category"] == cat][col].dropna().values
                      for cat in CATEGORY_ORDER]
            bp = ax.boxplot(groups, positions=range(len(CATEGORY_ORDER)),
                            patch_artist=True, widths=0.5,
                            medianprops=dict(color="black", lw=2))
            for patch, cat in zip(bp["boxes"], CATEGORY_ORDER):
                patch.set_facecolor(CATEGORY_COLORS[cat]); patch.set_alpha(0.7)
            for i, (grp, cat) in enumerate(zip(groups, CATEGORY_ORDER)):
                jitter = np.random.default_rng(i).uniform(-0.12, 0.12, len(grp))
                ax.scatter(i + jitter, grp, color=CATEGORY_COLORS[cat],
                           s=30, zorder=5, edgecolors="white", linewidths=0.5, alpha=0.9)
            valid_groups = [g for g in groups if len(g) > 0]
            kw_stat, kw_p = stats.kruskal(*valid_groups)
            ax.set_title(f"{label}\nKruskal-Wallis H={kw_stat:.2f}, p={kw_p:.3f}")
            ax.set_xticks(range(len(CATEGORY_ORDER)))
            ax.set_xticklabels([SHORT_CAT.get(c, c) for c in CATEGORY_ORDER],
                               rotation=15, ha="right")
            ax.set_ylabel(label)
            y_max = max(g.max() for g in groups if len(g) > 0)
            y_step = (y_max - ax.get_ylim()[0]) * 0.08
            for (i, j) in combinations(range(len(CATEGORY_ORDER)), 2):
                if len(groups[i]) < 2 or len(groups[j]) < 2: continue
                _, p_mw = stats.mannwhitneyu(groups[i], groups[j], alternative="two-sided")
                if p_mw < 0.10:
                    sig = "**" if p_mw < 0.01 else ("*" if p_mw < 0.05 else "†")
                    y_ann = y_max + y_step * (1 + abs(i - j))
                    ax.plot([i, j], [y_ann, y_ann], color="#2c3e50", lw=0.8)
                    ax.text((i + j) / 2, y_ann + y_step * 0.3, sig,
                            ha="center", color="#2c3e50")
        plt.tight_layout(); plt.show()
        fig.savefig(FIG_DIR / f"{fig_prefix}_group_comparison.png", dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"\nMean ± SD by category ({condition_name}):")
        display(df_cond.groupby("category")[list(metrics.keys())].agg(["mean", "std", "count"]).round(2))

    # ── 5. Individual agency items ─────────────────────────────────────────────
    q_labels = {k: v for k, v in {
        "agency_q1": "Q1 (control)",
        "agency_q2": "Q2 (intention)",
        "agency_q3": "Q3 (voluntary)",
    }.items() if k in acols}

    if not q_labels:
        print(f"\nIndividual Q items not available for {condition_name} condition.")
    else:
        fig, axes = plt.subplots(1, len(q_labels), figsize=(5 * len(q_labels), 5))
        if len(q_labels) == 1: axes = [axes]
        fig.suptitle(f"{condition_name}: individual agency items vs influence score")
        for ax, (col, label) in zip(axes, q_labels.items()):
            sub = df_cond.dropna(subset=[col, "influence_score"])
            for cat, grp in sub.groupby("category"):
                ax.scatter(grp[col], grp["influence_score"],
                           color=CATEGORY_COLORS.get(cat, "#95a5a6"),
                           label=cat, s=55, zorder=3, edgecolors="white", linewidths=0.5)
            for _, row in sub.iterrows():
                ax.annotate(row["pid"], (row[col], row["influence_score"]),
                            fontsize=10, ha="left", va="bottom",
                            xytext=(3, 2), textcoords="offset points", color="#555")
            x, y = sub[col].values.astype(float), sub["influence_score"].values
            rho, p_sp = stats.spearmanr(x, y)
            m, b, _, _, _ = stats.linregress(x, y)
            xfit = np.linspace(x.min(), x.max(), 100)
            ax.plot(xfit, m * xfit + b, color="#2c3e50", lw=1.2, linestyle="--")
            ax.set_title(f"{label}\nSpearman ρ={rho:.2f}, p={p_sp:.3f}")
            ax.set_xlabel(label); ax.set_ylabel("Influence score")
            ax.legend(fontsize=10, frameon=True)
        plt.tight_layout(); plt.show()
        fig.savefig(FIG_DIR / f"{fig_prefix}_individual_items.png", dpi=150, bbox_inches="tight")
        plt.close(fig)

---
## Part 1 — Weight Manipulation

The **weight** condition varied the heel/toe audio emphasis to shift participants' foot strike pattern. Directions: *increasing* (more heel emphasis) or *decreasing* (more toe emphasis).

In [ ]:
analyse_condition(
    df_cond         = df_weight,
    condition_name  = "Weight",
    direction_values= ["increasing", "decreasing"],
    fig_prefix      = "weight",
)

---
## Part 2 — Tempo Manipulation

The **tempo** condition varied walking cadence via a sigmoid ramp in the beat frequency. Directions: *speeding up* or *slowing down*.

Unlike the weight condition, all three individual agency Q items (Q1–Q3) are available in the tempo export.

In [ ]:
analyse_condition(
    df_cond         = df_tempo,
    condition_name  = "Tempo",
    direction_values= ["speeding up", "slowing down"],
    fig_prefix      = "tempo",
)

---
## Cross-condition comparison — Weight vs Tempo

Do participants show consistent influence scores across both conditions? Is agency predictive of influence regardless of condition type?

In [ ]:
# Merge on pid to get within-subject pairs
df_w = df_weight[["pid", "influence_score", "agency_aggregate"]].rename(
    columns={"influence_score": "w_influence", "agency_aggregate": "agency_agg"})
df_t2 = df_tempo[["pid", "influence_score"]].rename(
    columns={"influence_score": "t_influence"})
paired = df_w.merge(df_t2, on="pid").dropna(subset=["w_influence", "t_influence"])
print(f"Paired participants (both conditions): {len(paired)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: weight vs tempo influence score
ax = axes[0]
ax.scatter(paired["w_influence"], paired["t_influence"],
           color="#2c3e50", s=60, zorder=3, edgecolors="white", linewidths=0.5)
for _, row in paired.iterrows():
    ax.annotate(row["pid"], (row["w_influence"], row["t_influence"]),
                fontsize=10, ha="left", va="bottom",
                xytext=(3, 2), textcoords="offset points", color="#555")
lim_max = max(paired["w_influence"].max(), paired["t_influence"].max()) * 1.1
ax.plot([0, lim_max], [0, lim_max], color="#bdc3c7", lw=1, linestyle="--", label="y = x")
rho, p_wt = stats.spearmanr(paired["w_influence"], paired["t_influence"])
ax.set_title(f"Weight vs tempo influence score\nSpearman ρ={rho:.2f}, p={p_wt:.3f}")
ax.set_xlabel("Weight influence score"); ax.set_ylabel("Tempo influence score")
ax.legend(frameon=False)

# Right: agency aggregate vs both conditions
ax = axes[1]
ax.scatter(paired["agency_agg"], paired["w_influence"],
           color="#27ae60", s=55, label="Weight condition",
           zorder=3, edgecolors="white", linewidths=0.5)
ax.scatter(paired["agency_agg"], paired["t_influence"],
           color="#8e44ad", s=55, label="Tempo condition",
           zorder=3, marker="^", edgecolors="white", linewidths=0.5)
for cond, col in [("w_influence", "#27ae60"), ("t_influence", "#8e44ad")]:
    x, y = paired["agency_agg"].values, paired[cond].values
    m, b, r, p_lin, _ = stats.linregress(x, y)
    xfit = np.linspace(x.min(), x.max(), 100)
    ax.plot(xfit, m * xfit + b, color=col, lw=1.2, linestyle="--")
    rho, p_sp = stats.spearmanr(x, y)
    cond_name = "Weight" if cond == "w_influence" else "Tempo"
    print(f"{cond_name}: agency_agg vs influence  ρ={rho:+.3f}  p={p_sp:.3f}")
ax.set_title("Agency aggregate vs influence\n(both conditions, same participants)")
ax.set_xlabel("Agency aggregate"); ax.set_ylabel("Influence score")
ax.legend(frameon=True)

plt.tight_layout(); plt.show()
fig.savefig(FIG_DIR / "cross_condition_comparison.png", dpi=150, bbox_inches="tight")
plt.close(fig)

## Summary

Key things to look for in the results above:

- **Positive ρ** between agency and influence → high-agency participants show *more* gait adaptation
- **Negative ρ** → high-agency participants resist the manipulation
- **Category differences** → whether the *Consistent influence* group scores systematically higher or lower on any agency metric
- **Cross-condition consistency** → whether a participant's gait response is similar in both weight and tempo conditions, and whether agency predicts adaptation across both

All figures are saved automatically to `exports/figures/`.